In [ ]:
import sys
import os
import random
import torch

repo_root = os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath("__file__"))))
sys.path.insert(0, repo_root)

from vae_generator import VAEGenerator

SEQ_LENGTH = 6
INPUT_DIM = SEQ_LENGTH * 20  # 120
AMINO_ACIDS = list("ACDEFGHIKLMNPQRSTVWY")


# --- One-hot encode ---
aa_index = {aa: i for i, aa in enumerate(AMINO_ACIDS)}

def encode(sequences, seq_length):
    tensor = torch.zeros(len(sequences), seq_length * 20)
    for i, seq in enumerate(sequences):
        for j, aa in enumerate(seq):
            tensor[i, j * 20 + aa_index[aa]] = 1.0
    return tensor







Generated 500 unique temporary peptides
Example sequences: ['LWSECM', 'ECDVHR', 'DSVPDY', 'TMCQED', 'YGYQGN']


In [ ]:
repo_root = os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath("__file__"))))
sys.path.insert(0, repo_root)

# Import the JSONDataLoader class from the dataloader_json.py in dataloader module
from peptide_pipeline.dataloader.dataloader_json import DataLoader as JSONDataLoader
# Example usage
json_loader = JSONDataLoader()
json_loader.load_data("ai_training_peptides.json",columns=["sequence", "length"])

data = json_loader.get_data()

# Use the loaded data to train your model 
sequences = data["sequence"].tolist()
lengths = data["length"].tolist()
sequences.sort(key=lambda x: len(x))  # Sort sequences by length
sequences = [seq for seq in sequences if len(seq) == SEQ_LENGTH]  # Filter by desired length
print(f"Loaded {len(sequences)} sequences from JSON file.")
one_hot = encode(sequences, SEQ_LENGTH)
# --- Initialize model ---
vae = VAEGenerator(input_dim=INPUT_DIM, latent_dim=32, hidden_dim=128)
print(f"Device:     {vae.device}")
print(f"Input dim:  {vae.input_dim}")
print(f"Latent dim: {vae.latent_dim}")
print(f"Hidden dim: {vae.hidden_dim}")

# --- Train ---
one_hot = one_hot.to(vae.device)
vae.train_model(one_hot, epochs=1000, batch_size=64, lr=1e-3)
print("Training complete!")

# --- Generate ---
generated = vae.generate_peptides(count=10)
amino_acid_set = set(AMINO_ACIDS)
unique_generated = set()
for i, pep in enumerate(generated):
    valid = all(c in amino_acid_set for c in pep) and len(pep) == SEQ_LENGTH
    status = "✓" if valid else "✗ INVALID"
    unique_generated.add(pep)
    print(f"Peptide {i+1:02d}: {pep}  {status}")

# --- Novelty + diversity check ---
training_set = set(sequences)
novel = [p for p in generated if p not in training_set]
print(f"\nNovel peptides  (not in training data): {len(novel)}/{len(generated)}")
print(f"Unique peptides (diversity check):      {len(unique_generated)}/{len(generated)}")

Loaded 240 sequences from JSON file.
Device:     cuda
Input dim:  120
Latent dim: 32
Hidden dim: 128
  Epoch 050/300 | Recon: 0.5420 | KL: 0.5190 | KL weight: 0.49
  Epoch 100/300 | Recon: 0.3077 | KL: 0.3858 | KL weight: 0.99
  Epoch 150/300 | Recon: 0.2426 | KL: 0.3804 | KL weight: 1.00
  Epoch 200/300 | Recon: 0.1782 | KL: 0.3758 | KL weight: 1.00
  Epoch 250/300 | Recon: 0.1668 | KL: 0.3753 | KL weight: 1.00
  Epoch 300/300 | Recon: 0.1694 | KL: 0.3693 | KL weight: 1.00
Training complete!
Peptide 01: RWFWWW  ✓
Peptide 02: KIKRWC  ✓
Peptide 03: KRWKKF  ✓
Peptide 04: KRIWQR  ✓
Peptide 05: WRPWRI  ✓
Peptide 06: KHRRWR  ✓
Peptide 07: RLRRRH  ✓
Peptide 08: RRKWWR  ✓
Peptide 09: KLILWR  ✓
Peptide 10: FRWGRR  ✓

Novel peptides  (not in training data): 9/10
Unique peptides (diversity check):      10/10
